# ANKA-50M — Pairwise Combination Experiments

> **Status:** RMW3 selection experiment

## Purpose

Train and compare RM3 (Reflect+Morph), RW3 (Reflect+WorldPairs), and MW3 (Morph+WorldPairs) with a RAM-resident deterministic data path.

## Execution guidance

Use an A100 80 GB runtime. Restore the natural and WorldPairs shards, run structural/gradient/causal tests, then train the three arms sequentially.

## Primary outputs

Three checkpoints, held-out pair metrics, natural validation loss, generations, and a final report.

## Reproducibility notes

- Code-cell outputs and execution counters are intentionally cleared in the public release.
- Seeds, persistent paths, and checkpoint-resume behavior remain in the code.
- Review path constants before execution; historical notebooks may reference earlier run directories.
- Execute sections from top to bottom unless a section explicitly describes an independent experiment arm.


## 1. Set up paths, deterministic execution, and experiment settings

Prepares or validates persistent data artifacts. Review the Drive paths before running; large corpus operations can take time and may replace files at the configured destination.


In [ ]:
# CELL 1 — Setup, Restore Drive shards, and experiment settings

import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "tokenizers>=0.15",
    "pandas>=2.0",
])

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False,
)

from collections import defaultdict, deque
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional

import gc
import hashlib
import json
import math
import os
import random
import shutil
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from tokenizers import Tokenizer


# ============================================================
# 1. DETERMINISTIC SETTINGS
# ============================================================

SEED = 3407

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

if DEVICE.type != "cuda":
    raise RuntimeError(
        "Runtime > Change runtime type > Select A100 GPU."
    )

if not torch.cuda.is_bf16_supported():
    raise RuntimeError(
        "This notebook requires a BF16 supported CUDA GPU."
    )

GPU_GIB = (
    torch.cuda.get_device_properties(0).total_memory
    / 2**30
)

if GPU_GIB < 70:
    raise RuntimeError(
        "Expected A100 80 GB. "
        f"Detected VRAM: {GPU_GIB:.2f} GiB"
    )

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")


# ============================================================
# 2. PERSISTENT DRIVE PATHS
# ============================================================

DRIVE_RUN = Path(
    "/content/drive/MyDrive/ANKA/run_01"
)

DRIVE_SHARDS = (
    DRIVE_RUN
    / "shards"
)

DRIVE_MANIFEST = (
    DRIVE_SHARDS
    / "manifest.json"
)

TOKENIZER_PATH = (
    DRIVE_RUN
    / "tokenizer"
    / "tokenizer.json"
)

WORLD_DRIVE_ROOT = (
    DRIVE_RUN
    / "worldpairs_proof_v1"
)

PROOF_SUITE_DRIVE = (
    WORLD_DRIVE_ROOT
    / "evaluation"
    / "proof.jsonl"
)

OUTPUT_ROOT = (
    DRIVE_RUN
    / "pairwise_combinations_v1"
)


# ============================================================
# 3. TEMPORARY COLAB PATHS
# ============================================================

LOCAL_SHARDS = Path(
    "/content/anka_shards"
)

NATURAL_MANIFEST = (
    LOCAL_SHARDS
    / "manifest.json"
)

LOCAL_CACHE = Path(
    "/content/anka_pairwise_cache"
)

LOCAL_SHARDS.mkdir(
    parents=True,
    exist_ok=True,
)

LOCAL_CACHE.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 4. FILE CHECKS
# ============================================================

assert DRIVE_MANIFEST.exists(), (
    "Tokenized manifest not found in Drive: "
    f"{DRIVE_MANIFEST}"
)

assert TOKENIZER_PATH.exists(), (
    f"Tokenizer not found: {TOKENIZER_PATH}"
)

assert PROOF_SUITE_DRIVE.exists(), (
    "WorldPairs proof test suite not found: "
    f"{PROOF_SUITE_DRIVE}"
)


# ============================================================
# 5. RESTORE DRIVE SHARDS TO /content
#
# Only missing or modified files are copied here.
# The next RAM-loader cell will fully load these local files
# into RAM. Drive or disk will not be read during training.
# ============================================================

def stage_natural_shards() -> int:
    manifest = json.loads(
        DRIVE_MANIFEST.read_text(
            encoding="utf-8"
        )
    )

    shard_entries = manifest.get(
        "shards",
        [],
    )

    if not isinstance(shard_entries, list):
        raise RuntimeError(
            "'shards' field in Drive manifest is not a list."
        )

    if not shard_entries:
        raise RuntimeError(
            "No shards found in Drive manifest."
        )

    transfer_plan = []

    for shard_index, item in enumerate(shard_entries):
        if not isinstance(item, dict):
            raise RuntimeError(
                f"Invalid shard entry: index={shard_index}"
            )

        if isinstance(item.get("file"), str):
            path_key = "file"
        elif isinstance(item.get("path"), str):
            path_key = "path"
        else:
            raise RuntimeError(
                "'file' or 'path' not found in shard entry: "
                f"index={shard_index}"
            )

        recorded_path = Path(
            item[path_key]
        )

        source_candidates = [
            recorded_path,
            DRIVE_SHARDS / recorded_path,
            DRIVE_SHARDS / recorded_path.name,
        ]

        source_path = next(
            (
                candidate
                for candidate in source_candidates
                if candidate.exists()
                and candidate.is_file()
            ),
            None,
        )

        if source_path is None:
            raise FileNotFoundError(
                "Drive shard file not found: "
                f"{recorded_path.name}"
            )

        destination_path = (
            LOCAL_SHARDS
            / source_path.name
        )

        transfer_plan.append(
            (
                source_path,
                destination_path,
            )
        )

        # Local manifest will only point to files under /content
        item[path_key] = source_path.name

    total_bytes = sum(
        source.stat().st_size
        for source, _destination in transfer_plan
    )

    copied_files = 0
    copied_bytes = 0

    print(
        f"Natural shard count: {len(transfer_plan):,}"
    )
    print(
        f"Total shard size: {total_bytes / 2**30:.2f} GiB"
    )

    for index, (source, destination) in enumerate(
        transfer_plan,
        start=1,
    ):
        copy_required = (
            not destination.exists()
            or destination.stat().st_size
            != source.stat().st_size
        )

        if copy_required:
            temporary = destination.with_suffix(
                destination.suffix + ".part"
            )

            shutil.copy2(
                source,
                temporary,
            )

            if (
                temporary.stat().st_size
                != source.stat().st_size
            ):
                raise IOError(
                    "Shard copy size check failed: "
                    f"{source.name}"
                )

            temporary.replace(
                destination
            )

            copied_files += 1
            copied_bytes += source.stat().st_size

        if (
            index % 10 == 0
            or index == len(transfer_plan)
        ):
            print(
                f"Preparing shards: "
                f"{index:,}/{len(transfer_plan):,} | "
                f"copied in this run="
                f"{copied_files:,} files, "
                f"{copied_bytes / 2**30:.2f} GiB",
                flush=True,
            )

    # Manifest is written after all shards are prepared.
    manifest_temporary = (
        NATURAL_MANIFEST
        .with_suffix(".tmp")
    )

    manifest_temporary.write_text(
        json.dumps(
            manifest,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    manifest_temporary.replace(
        NATURAL_MANIFEST
    )

    # Final verification
    for _source, destination in transfer_plan:
        if not destination.exists():
            raise FileNotFoundError(
                f"Local shard missing: {destination}"
            )

    return len(transfer_plan)


RESTORED_SHARD_COUNT = stage_natural_shards()

assert NATURAL_MANIFEST.exists(), (
    "Failed to create local manifest: "
    f"{NATURAL_MANIFEST}"
)


# ============================================================
# 6. TRAINING SETTINGS
# ============================================================

SEQ_LEN = 1_024

TARGET_GLOBAL_BATCH = 64

# If later actual VRAM test is required, it will automatically
# convert this to 32 x 2 or 16 x 4 format.
MICRO_BATCH = TARGET_GLOBAL_BATCH
GRAD_ACCUM = 1

TOKEN_BUDGET = 256_000_000
WORLD_RATIO = 0.10


# ============================================================
# 7. UTILITY FUNCTIONS
# ============================================================

def seed_everything(
    seed: int,
) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def clear_cuda() -> None:
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


seed_everything(SEED)


# ============================================================
# 8. SUMMARY
# ============================================================

print()
print("=" * 72)
print("SETUP COMPLETED")
print("=" * 72)
print("GPU             :", torch.cuda.get_device_name(0))
print("VRAM            :", f"{GPU_GIB:.2f} GiB")
print("Drive manifest  :", DRIVE_MANIFEST)
print("Local manifest  :", NATURAL_MANIFEST)
print("Prepared shards :", RESTORED_SHARD_COUNT)
print("Tokenizer       :", TOKENIZER_PATH)
print("Sequence        :", SEQ_LEN)
print("Target batch    :", TARGET_GLOBAL_BATCH)
print("Token/arm       :", f"{TOKEN_BUDGET:,}")
print("Output folder   :", OUTPUT_ROOT)
print()
print(
    "The next RAM-loader cell will load these shards "
    "completely into system RAM."
)


## 2. Stage WorldPairs locally and verify natural shards

Prepares or validates persistent data artifacts. Review the Drive paths before running; large corpus operations can take time and may replace files at the configured destination.


In [ ]:
# CELL 2 — Stage the small WorldPairs corpus locally; natural shards must already be local
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_world_manifest() -> Path:
    candidates = sorted(WORLD_DRIVE_ROOT.glob(f"corpus_*_seq{SEQ_LEN}/manifest.json"))
    valid = []
    for path in candidates:
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
            entry = payload["corpora"]["paired"]
            if (
                payload.get("format") == "worldpairs-fixed-uint16-v1"
                and int(payload["sequence_length"]) == SEQ_LEN
            ):
                valid.append((int(entry["predicted_tokens"]), path))
        except Exception:
            continue
    if not valid:
        raise FileNotFoundError(
            f"SEQ_LEN={SEQ_LEN} for WorldPairs manifest not found: {WORLD_DRIVE_ROOT}"
        )
    return max(valid, key=lambda item: item[0])[1]


def stage_worldpairs(source_manifest: Path) -> tuple[Path, Path]:
    payload = json.loads(source_manifest.read_text(encoding="utf-8"))
    entry = payload["corpora"]["paired"]
    local_dir = LOCAL_CACHE / "worldpairs"
    local_dir.mkdir(parents=True, exist_ok=True)
    for key, hash_key in (("tokens_file", "sha256_tokens"), ("lengths_file", "sha256_lengths")):
        source = source_manifest.parent / entry[key]
        target = local_dir / source.name
        expected_hash = entry.get(hash_key)
        valid = (
            target.exists()
            and target.stat().st_size == source.stat().st_size
            and (expected_hash is None or sha256_file(target) == expected_hash)
        )
        if not valid:
            print("Yerel onbellege copying:", source.name)
            temporary = target.with_suffix(target.suffix + ".tmp")
            shutil.copy2(source, temporary)
            temporary.replace(target)
        if expected_hash is not None:
            assert sha256_file(target) == expected_hash, f"Hash error: {target}"

    local_manifest = local_dir / "manifest.json"
    local_manifest.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    local_suite = local_dir / "proof.jsonl"
    if (
        not local_suite.exists()
        or local_suite.stat().st_size != PROOF_SUITE_DRIVE.stat().st_size
    ):
        shutil.copy2(PROOF_SUITE_DRIVE, local_suite)
    return local_manifest, local_suite


WORLD_MANIFEST_DRIVE = find_world_manifest()
WORLD_MANIFEST, PROOF_SUITE = stage_worldpairs(WORLD_MANIFEST_DRIVE)
print("World manifest:", WORLD_MANIFEST_DRIVE)
print("Yerel manifest:", WORLD_MANIFEST)
print("Proof suite    :", PROOF_SUITE)


## 3. Define shared Transformer components

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# CELL 3 — Shared transformer components
@dataclass
class ANKAConfig:
    name: str
    vocab_size: int = 15_360
    max_seq_len: int = 2_048
    d_model: int = 448
    n_layers: int = 20
    n_heads: int = 7
    n_kv_heads: int = 1
    d_ff: int = 1_216
    rope_theta: float = 10_000.0
    norm_eps: float = 1e-5
    dropout: float = 0.0


@dataclass
class ANKAOutput:
    logits: torch.Tensor
    loss: Optional[torch.Tensor] = None


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        dtype = x.dtype
        y = x.float()
        y = y * torch.rsqrt(y.square().mean(dim=-1, keepdim=True) + self.eps)
        return y.to(dtype) * self.weight.to(dtype=dtype)


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim: int, max_seq_len: int, theta: float):
        super().__init__()
        assert head_dim % 2 == 0
        inv = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        positions = torch.arange(max_seq_len).float()
        angles = torch.outer(positions, inv)
        self.register_buffer("cos", angles.cos(), persistent=False)
        self.register_buffer("sin", angles.sin(), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        sequence = x.shape[-2]
        cos = self.cos[:sequence].to(device=x.device, dtype=x.dtype)[None, None]
        sin = self.sin[:sequence].to(device=x.device, dtype=x.dtype)[None, None]
        even, odd = x[..., 0::2], x[..., 1::2]
        result = torch.empty_like(x)
        result[..., 0::2] = even * cos - odd * sin
        result[..., 1::2] = even * sin + odd * cos
        return result


class GQAttention(nn.Module):
    def __init__(self, cfg: ANKAConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        assert cfg.n_heads % cfg.n_kv_heads == 0
        self.n_heads = cfg.n_heads
        self.n_kv_heads = cfg.n_kv_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.dropout = cfg.dropout
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(cfg.n_heads * self.head_dim, cfg.d_model, bias=False)
        self.rope = RotaryEmbedding(self.head_dim, cfg.max_seq_len, cfg.rope_theta)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, sequence, _ = x.shape
        q = self.q_proj(x).view(batch, sequence, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch, sequence, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch, sequence, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q, k = self.rope(q), self.rope(k)
        y = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
            enable_gqa=(self.n_heads != self.n_kv_heads),
        )
        return self.o_proj(y.transpose(1, 2).contiguous().view(batch, sequence, -1))


class SwiGLU(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down(F.silu(self.gate(x)) * self.up(x))


def causal_lm_loss(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    return F.cross_entropy(
        logits[:, :-1].contiguous().view(-1, logits.size(-1)),
        labels[:, 1:].contiguous().view(-1),
        ignore_index=-100,
    )


def cuda_autocast_enabled() -> bool:
    try:
        return torch.is_autocast_enabled("cuda")
    except TypeError:
        return torch.is_autocast_enabled()


def initialize_model(model: nn.Module, std: float = 0.02) -> None:
    for module in model.modules():
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=std)


class SparseMorphFFN(nn.Module):
    def __init__(self, d_model: int, n_experts: int):
        super().__init__()
        self.router = nn.Linear(d_model, n_experts, bias=False)
        self.n_experts = n_experts
        self.last_balance = None
        self.last_entropy = None
        self.last_load = None

    def forward(self, x: torch.Tensor, experts: nn.ModuleList) -> torch.Tensor:
        shape = x.shape
        flat = x.reshape(-1, shape[-1])
        router_logits = self.router(flat).float()
        probs = torch.softmax(router_logits, dim=-1)
        choice = probs.argmax(dim=-1)
        out = torch.zeros_like(flat)
        for expert_id, expert in enumerate(experts):
            indices = torch.nonzero(choice == expert_id, as_tuple=False).flatten()
            if indices.numel() == 0:
                continue
            selected = flat.index_select(0, indices)
            expert_out = expert(selected)
            p = probs.index_select(0, indices)[:, expert_id].to(expert_out.dtype)
            straight_through = 1.0 + p - p.detach()
            dispatched = expert_out * straight_through.unsqueeze(-1)
            out = out.index_copy(0, indices, dispatched.to(dtype=out.dtype))
        importance = probs.mean(dim=0)
        load = F.one_hot(choice, self.n_experts).float().mean(dim=0)
        self.last_balance = self.n_experts * torch.sum(importance * load)
        self.last_entropy = -(probs * probs.clamp_min(1e-9).log()).sum(-1).mean()
        self.last_load = load.detach()
        return out.view(shape)


## 4. Define the RW3 and MW3 pairwise architectures

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# CELL 4 — R3+W3 and M3+W3 models
class ReflectiveBlock(nn.Module):
    def __init__(self, cfg: ANKAConfig):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = GQAttention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ffn = SwiGLU(cfg.d_model, cfg.d_ff)

    def forward(
        self,
        x: torch.Tensor,
        reflection: Optional[torch.Tensor],
        reflection_gate: Optional[torch.Tensor],
        residual_scale: float,
    ) -> torch.Tensor:
        conditioned = x if reflection is None else x + reflection_gate * reflection
        x = x + residual_scale * self.attn(self.attn_norm(conditioned))
        conditioned = x if reflection is None else x + reflection_gate * reflection
        x = x + residual_scale * self.ffn(self.ffn_norm(conditioned))
        return x


class ANKAReflectiveLoop(nn.Module):
    def __init__(self, cfg: ANKAConfig, draft_weight: float = 0.25, aux_stride: int = 8):
        super().__init__()
        self.config = cfg
        self.draft_weight = draft_weight
        self.aux_stride = aux_stride
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([ReflectiveBlock(cfg) for _ in range(cfg.n_layers)])
        self.round_embedding = nn.Parameter(torch.empty(2, cfg.d_model))
        self.reflect_in_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.reflect_delta_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.reflect_proj = nn.Linear(2 * cfg.d_model, cfg.d_model, bias=False)
        self.reflect_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.reflect_gate_logits = nn.Parameter(torch.full((cfg.n_layers,), -2.1972246))
        self.draft_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.final_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.residual_scale = 1.0 / math.sqrt(2.0 * cfg.n_layers)
        self.last_metrics = {}
        initialize_model(self)
        nn.init.normal_(self.round_embedding, std=0.002)
        nn.init.normal_(self.reflect_proj.weight, std=0.01)

    def num_trainable_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward(self, input_ids: torch.Tensor, labels: Optional[torch.Tensor] = None) -> ANKAOutput:
        if input_ids.shape[1] > self.config.max_seq_len:
            raise ValueError("Sequence max_seq_len limit exceeds")
        token_state = self.embedding(input_ids)
        if token_state.is_cuda and cuda_autocast_enabled():
            token_state = token_state.to(torch.bfloat16)
        h = token_state + self.round_embedding[0].to(token_state.dtype)
        for block in self.blocks:
            h = block(h, None, None, self.residual_scale)
        draft = h
        discrepancy = self.reflect_delta_norm(draft - token_state)
        reflection = torch.tanh(self.reflect_proj(torch.cat([
            self.reflect_in_norm(draft), discrepancy
        ], dim=-1)))
        reflection = self.reflect_norm(reflection)
        h = draft + self.round_embedding[1].to(draft.dtype)
        gates = torch.sigmoid(self.reflect_gate_logits)
        for layer_id, block in enumerate(self.blocks):
            h = block(h, reflection, gates[layer_id].to(h.dtype), self.residual_scale)
        logits = F.linear(self.final_norm(h), self.embedding.weight)
        final_loss = draft_loss = loss = None
        if labels is not None:
            final_loss = causal_lm_loss(logits, labels)
            if self.training and self.draft_weight > 0:
                draft_hidden = self.draft_norm(draft)[:, :-1:self.aux_stride]
                draft_targets = labels[:, 1::self.aux_stride]
                draft_logits = F.linear(draft_hidden, self.embedding.weight)
                draft_loss = F.cross_entropy(
                    draft_logits.reshape(-1, draft_logits.size(-1)),
                    draft_targets.reshape(-1),
                    ignore_index=-100,
                )
                loss = final_loss + self.draft_weight * draft_loss
            else:
                loss = final_loss
        self.last_metrics = {
            "final_loss": None if final_loss is None else final_loss.detach(),
            "draft_loss": None if draft_loss is None else draft_loss.detach(),
            "reflect_gate": gates.mean().detach(),
            "reflection_rms": reflection.float().square().mean().sqrt().detach(),
        }
        return ANKAOutput(logits=logits, loss=loss)


class MorphBlock(nn.Module):
    def __init__(self, cfg: ANKAConfig, n_experts: int):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = GQAttention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.morph = SparseMorphFFN(cfg.d_model, n_experts)
        self.residual_scale = 1.0 / math.sqrt(2.0 * cfg.n_layers)

    def forward(self, x: torch.Tensor, experts: nn.ModuleList) -> torch.Tensor:
        x = x + self.residual_scale * self.attn(self.attn_norm(x))
        x = x + self.residual_scale * self.morph(self.ffn_norm(x), experts)
        return x


class ANKASparseMorph(nn.Module):
    def __init__(self, cfg: ANKAConfig, n_stages: int = 5, experts_per_stage: int = 4):
        super().__init__()
        assert cfg.n_layers % n_stages == 0
        self.config = cfg
        self.n_stages = n_stages
        self.experts_per_stage = experts_per_stage
        self.layers_per_stage = cfg.n_layers // n_stages
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.stage_experts = nn.ModuleList([
            nn.ModuleList([
                SwiGLU(cfg.d_model, cfg.d_ff) for _ in range(experts_per_stage)
            ]) for _ in range(n_stages)
        ])
        self.blocks = nn.ModuleList([
            MorphBlock(cfg, experts_per_stage) for _ in range(cfg.n_layers)
        ])
        self.final_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.balance_weight = 0.01
        self.last_metrics = {}
        initialize_model(self)

    def num_trainable_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward(self, input_ids: torch.Tensor, labels: Optional[torch.Tensor] = None) -> ANKAOutput:
        if input_ids.shape[1] > self.config.max_seq_len:
            raise ValueError("Sequence max_seq_len limit exceeds")
        h = self.embedding(input_ids)
        if h.is_cuda and cuda_autocast_enabled():
            h = h.to(torch.bfloat16)
        balances, entropies = [], []
        stage_loads = [[] for _ in range(self.n_stages)]
        for layer_id, block in enumerate(self.blocks):
            stage_id = layer_id // self.layers_per_stage
            h = block(h, self.stage_experts[stage_id])
            balances.append(block.morph.last_balance)
            entropies.append(block.morph.last_entropy)
            stage_loads[stage_id].append(block.morph.last_load)
        logits = F.linear(self.final_norm(h), self.embedding.weight)
        balance_loss = torch.stack(balances).mean()
        lm_loss = causal_lm_loss(logits, labels) if labels is not None else None
        loss = None if lm_loss is None else lm_loss + self.balance_weight * balance_loss
        self.last_metrics = {
            "lm_loss": None if lm_loss is None else lm_loss.detach(),
            "balance_loss": balance_loss.detach(),
            "router_entropy": torch.stack(entropies).mean().detach(),
            "stage_loads": torch.stack([
                torch.stack(items).mean(0) for items in stage_loads
            ]).detach(),
        }
        return ANKAOutput(logits=logits, loss=loss)


def build_rw3() -> ANKAReflectiveLoop:
    cfg = ANKAConfig(
        name="RW3", d_model=512, n_layers=13,
        n_heads=8, n_kv_heads=2, d_ff=1536,
    )
    return ANKAReflectiveLoop(cfg)


def build_mw3() -> ANKASparseMorph:
    cfg = ANKAConfig(
        name="MW3", d_model=448, n_layers=20,
        n_heads=7, n_kv_heads=1, d_ff=1216,
    )
    return ANKASparseMorph(cfg, n_stages=5, experts_per_stage=4)


## 5. Define the RM3 Reflect+Morph architecture

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# CELL 5 — R3+M3 architecture under the 50M parameter limit
class ReflectiveMorphBlock(nn.Module):
    def __init__(self, cfg: ANKAConfig, n_experts: int):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = GQAttention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.morph = SparseMorphFFN(cfg.d_model, n_experts)

    def forward(
        self,
        x: torch.Tensor,
        experts: nn.ModuleList,
        reflection: Optional[torch.Tensor],
        reflection_gate: Optional[torch.Tensor],
        residual_scale: float,
    ) -> torch.Tensor:
        conditioned = x if reflection is None else x + reflection_gate * reflection
        x = x + residual_scale * self.attn(self.attn_norm(conditioned))
        conditioned = x if reflection is None else x + reflection_gate * reflection
        x = x + residual_scale * self.morph(self.ffn_norm(conditioned), experts)
        return x


class ANKAReflectiveMorph(nn.Module):
    def __init__(
        self,
        cfg: ANKAConfig,
        n_stages: int = 4,
        experts_per_stage: int = 4,
        draft_weight: float = 0.25,
        balance_weight: float = 0.01,
        aux_stride: int = 8,
    ):
        super().__init__()
        self.config = cfg
        self.n_stages = n_stages
        self.experts_per_stage = experts_per_stage
        self.draft_weight = draft_weight
        self.balance_weight = balance_weight
        self.aux_stride = aux_stride
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.stage_experts = nn.ModuleList([
            nn.ModuleList([
                SwiGLU(cfg.d_model, cfg.d_ff) for _ in range(experts_per_stage)
            ]) for _ in range(n_stages)
        ])
        self.blocks = nn.ModuleList([
            ReflectiveMorphBlock(cfg, experts_per_stage)
            for _ in range(cfg.n_layers)
        ])
        self.round_embedding = nn.Parameter(torch.empty(2, cfg.d_model))
        self.reflect_in_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.reflect_delta_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.reflect_proj = nn.Linear(2 * cfg.d_model, cfg.d_model, bias=False)
        self.reflect_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.reflect_gate_logits = nn.Parameter(torch.full((cfg.n_layers,), -2.1972246))
        self.draft_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.final_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.residual_scale = 1.0 / math.sqrt(2.0 * cfg.n_layers)
        self.last_metrics = {}
        initialize_model(self)
        nn.init.normal_(self.round_embedding, std=0.002)
        nn.init.normal_(self.reflect_proj.weight, std=0.01)

    def stage_id(self, layer_id: int) -> int:
        return min(self.n_stages - 1, layer_id * self.n_stages // self.config.n_layers)

    def num_trainable_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward(self, input_ids: torch.Tensor, labels: Optional[torch.Tensor] = None) -> ANKAOutput:
        if input_ids.shape[1] > self.config.max_seq_len:
            raise ValueError("Sequence max_seq_len limit exceeds")
        token_state = self.embedding(input_ids)
        if token_state.is_cuda and cuda_autocast_enabled():
            token_state = token_state.to(torch.bfloat16)

        balances, entropies = [], []
        stage_loads = [[] for _ in range(self.n_stages)]
        h = token_state + self.round_embedding[0].to(token_state.dtype)
        for layer_id, block in enumerate(self.blocks):
            stage = self.stage_id(layer_id)
            h = block(h, self.stage_experts[stage], None, None, self.residual_scale)
            balances.append(block.morph.last_balance)
            entropies.append(block.morph.last_entropy)
            stage_loads[stage].append(block.morph.last_load)
        draft = h

        discrepancy = self.reflect_delta_norm(draft - token_state)
        reflection = torch.tanh(self.reflect_proj(torch.cat([
            self.reflect_in_norm(draft), discrepancy
        ], dim=-1)))
        reflection = self.reflect_norm(reflection)
        h = draft + self.round_embedding[1].to(draft.dtype)
        gates = torch.sigmoid(self.reflect_gate_logits)
        for layer_id, block in enumerate(self.blocks):
            stage = self.stage_id(layer_id)
            h = block(
                h, self.stage_experts[stage], reflection,
                gates[layer_id].to(h.dtype), self.residual_scale,
            )
            balances.append(block.morph.last_balance)
            entropies.append(block.morph.last_entropy)
            stage_loads[stage].append(block.morph.last_load)

        logits = F.linear(self.final_norm(h), self.embedding.weight)
        balance_loss = torch.stack(balances).mean()
        final_loss = draft_loss = loss = None
        if labels is not None:
            final_loss = causal_lm_loss(logits, labels)
            if self.training and self.draft_weight > 0:
                draft_hidden = self.draft_norm(draft)[:, :-1:self.aux_stride]
                draft_targets = labels[:, 1::self.aux_stride]
                draft_logits = F.linear(draft_hidden, self.embedding.weight)
                draft_loss = F.cross_entropy(
                    draft_logits.reshape(-1, draft_logits.size(-1)),
                    draft_targets.reshape(-1),
                    ignore_index=-100,
                )
                loss = (
                    final_loss
                    + self.draft_weight * draft_loss
                    + self.balance_weight * balance_loss
                )
            else:
                loss = final_loss
        self.last_metrics = {
            "final_loss": None if final_loss is None else final_loss.detach(),
            "draft_loss": None if draft_loss is None else draft_loss.detach(),
            "balance_loss": balance_loss.detach(),
            "reflect_gate": gates.mean().detach(),
            "reflection_rms": reflection.float().square().mean().sqrt().detach(),
            "router_entropy": torch.stack(entropies).mean().detach(),
            "stage_loads": torch.stack([
                torch.stack(items).mean(0) for items in stage_loads
            ]).detach(),
        }
        return ANKAOutput(logits=logits, loss=loss)


def build_rm3() -> ANKAReflectiveMorph:
    cfg = ANKAConfig(
        name="RM3", d_model=512, n_layers=13,
        n_heads=8, n_kv_heads=2, d_ff=1280,
    )
    return ANKAReflectiveMorph(cfg, n_stages=4, experts_per_stage=4)


BUILDERS = {"RM3": build_rm3, "RW3": build_rw3, "MW3": build_mw3}
EXPECTED_PARAMETERS = {
    "RM3": 48_409_101,
    "RW3": 47_596_045,
    "MW3": 48_796_608,
}


## 6. Run structural, gradient, causal, and VRAM tests

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# CELL 6 — structural, gradient, causal and VRAM tests

def structural_test(arm: str) -> dict:
    clear_cuda()
    seed_everything(SEED)

    model = BUILDERS[arm]().to(DEVICE)

    parameters = model.num_trainable_parameters()

    assert parameters == EXPECTED_PARAMETERS[arm], (
        f"{arm}: parameter count hatali: "
        f"{parameters:,} != {EXPECTED_PARAMETERS[arm]:,}"
    )

    assert parameters < 50_000_000, (
        f"{arm}: 50M parameter limit was-exceeded"
    )

    # ==========================================================
    # 1. FORWARD + LOSS + GRADIENT TESTI
    # ==========================================================

    model.train()

    input_ids = torch.randint(
        low=0,
        high=model.config.vocab_size,
        size=(2, 32),
        device=DEVICE,
    )

    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
    ):
        output = model(
            input_ids,
            labels=input_ids,
        )

    expected_shape = (
        2,
        32,
        model.config.vocab_size,
    )

    assert output.logits.shape == expected_shape, (
        f"{arm}: logits shape hatali: "
        f"{tuple(output.logits.shape)}"
    )

    assert torch.isfinite(output.loss), (
        f"{arm}: loss is not finite"
    )

    output.loss.backward()

    loss_value = float(
        output.loss.detach().float().cpu()
    )

    required_terms = {
        "RM3": ("reflect_proj", "router"),
        "RW3": ("reflect_proj",),
        "MW3": ("router",),
    }[arm]

    for term in required_terms:
        matched_parameters = [
            (name, parameter)
            for name, parameter in model.named_parameters()
            if term in name
        ]

        assert matched_parameters, (
            f"{arm}: {term} parameter not found"
        )

        bad_gradients = [
            name
            for name, parameter in matched_parameters
            if (
                parameter.grad is None
                or not torch.isfinite(
                    parameter.grad
                ).all()
            )
        ]

        assert not bad_gradients, (
            f"{arm}: gradient error: "
            f"{bad_gradients[:5]}"
        )

    model.zero_grad(set_to_none=True)

    del output
    del input_ids

    clear_cuda()

    # ==========================================================
    # 2. CAUSAL MASK TESTI
    # ==========================================================

    model.eval()

    prefix_length = 13

    sequence_a = torch.randint(
        low=0,
        high=model.config.vocab_size,
        size=(1, 24),
        device=DEVICE,
    )

    sequence_b = sequence_a.clone()

    # future tokenlarin definitely different to-be ensure.
    future_offsets = torch.randint(
        low=1,
        high=model.config.vocab_size,
        size=sequence_b[:, prefix_length:].shape,
        device=DEVICE,
    )

    sequence_b[:, prefix_length:] = (
        sequence_b[:, prefix_length:]
        + future_offsets
    ) % model.config.vocab_size

    previous_matmul_tf32 = (
        torch.backends.cuda.matmul.allow_tf32
    )

    previous_cudnn_tf32 = (
        torch.backends.cudnn.allow_tf32
    )

    previous_flash = (
        torch.backends.cuda.flash_sdp_enabled()
    )

    previous_mem_efficient = (
        torch.backends.cuda.mem_efficient_sdp_enabled()
    )

    previous_math = (
        torch.backends.cuda.math_sdp_enabled()
    )

    try:
        # Disable approximate TF32 arithmetic during the causal test.
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False

        # Kararli FP32 attention hesabi.
        torch.backends.cuda.enable_flash_sdp(False)
        torch.backends.cuda.enable_mem_efficient_sdp(False)
        torch.backends.cuda.enable_math_sdp(True)

        with torch.inference_mode():
            logits_a = model(sequence_a).logits[
                :, :prefix_length
            ].float()

            logits_b = model(sequence_b).logits[
                :, :prefix_length
            ].float()

        torch.cuda.synchronize()

        causal_error = float(
            (
                logits_a
                - logits_b
            )
            .abs()
            .max()
            .item()
        )

    finally:
        torch.backends.cuda.matmul.allow_tf32 = (
            previous_matmul_tf32
        )

        torch.backends.cudnn.allow_tf32 = (
            previous_cudnn_tf32
        )

        torch.backends.cuda.enable_flash_sdp(
            previous_flash
        )

        torch.backends.cuda.enable_mem_efficient_sdp(
            previous_mem_efficient
        )

        torch.backends.cuda.enable_math_sdp(
            previous_math
        )

    # Differences around 9.53e-7 are normal FP32 rounding error.
    # 1e-4 actual sizintiyi yakalamak for sufficiently strict.
    causal_tolerance = 1e-4

    assert causal_error < causal_tolerance, (
        f"{arm}: causal leakage found: "
        f"{causal_error:.10g}"
    )

    result = {
        "arm": arm,
        "parameters": parameters,
        "loss": loss_value,
        "causal_error": causal_error,
        "causal_tolerance": causal_tolerance,
        "pass": True,
    }

    print(
        f"{arm}: "
        f"parameters={parameters:,} | "
        f"loss={loss_value:.4f} | "
        f"causal_error={causal_error:.10g} | "
        f"PASS"
    )

    del model
    del sequence_a
    del sequence_b
    del future_offsets
    del logits_a
    del logits_b

    clear_cuda()

    return result


STRUCTURAL_RESULTS = [
    structural_test(arm)
    for arm in BUILDERS
]

print("\nAll structural tests PASS")


# ==============================================================
# 3. shared MICRO-BATCH selection
# ==============================================================

def select_training_geometry() -> tuple[int, int]:
    """
    Optimizer adimi dahil all modellerde working
    Selects the largest shared micro-batch value.
    """

    candidates = (
        64,
        32,
        16,
    )

    for candidate in candidates:
        if TARGET_GLOBAL_BATCH % candidate != 0:
            continue

        all_models_fit = True

        print(
            f"\nVRAM testi: micro_batch={candidate}"
        )

        for arm in BUILDERS:
            model = None
            optimizer = None
            input_ids = None
            output = None
            loss = None

            try:
                clear_cuda()
                seed_everything(SEED)

                model = BUILDERS[arm]().to(DEVICE)
                model.train()

                optimizer = torch.optim.AdamW(
                    model.parameters(),
                    lr=1e-4,
                    betas=(0.9, 0.95),
                    eps=1e-8,
                    weight_decay=0.10,
                    fused=True,
                )

                input_ids = torch.randint(
                    low=0,
                    high=model.config.vocab_size,
                    size=(
                        candidate,
                        SEQ_LEN,
                    ),
                    device=DEVICE,
                )

                torch.cuda.reset_peak_memory_stats()

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.bfloat16,
                ):
                    output = model(
                        input_ids,
                        labels=input_ids,
                    )

                    loss = output.loss

                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0,
                )

                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

                torch.cuda.synchronize()

                peak_vram_gib = (
                    torch.cuda.max_memory_allocated()
                    / 2**30
                )

                print(
                    f"  {arm}: FIT | "
                    f"peak={peak_vram_gib:.2f} GiB"
                )

            except torch.cuda.OutOfMemoryError:
                print(f"  {arm}: OOM")
                all_models_fit = False

            except RuntimeError as error:
                if "out of memory" not in str(error).lower():
                    raise

                print(f"  {arm}: OOM")
                all_models_fit = False

            finally:
                if optimizer is not None:
                    optimizer.zero_grad(
                        set_to_none=True
                    )

                del model
                del optimizer
                del input_ids
                del output
                del loss

                clear_cuda()

            if not all_models_fit:
                break

        if all_models_fit:
            gradient_accumulation = (
                TARGET_GLOBAL_BATCH
                // candidate
            )

            return (
                candidate,
                gradient_accumulation,
            )

    raise RuntimeError(
        "Micro-batch 16 bile sigmadi."
    )


MICRO_BATCH, GRAD_ACCUM = (
    select_training_geometry()
)

print(
    "\nSecilen micro/accum:",
    MICRO_BATCH,
    "/",
    GRAD_ACCUM,
)

## 7. Build the deterministic vectorized RAM data pipeline

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# CELL 7 — Fast local, vectorized, deterministic batch pipeline
def resolve_path(root: Path, value: str) -> Optional[Path]:
    raw = Path(value)
    for candidate in (raw, root / raw, root / raw.name):
        if candidate.exists():
            return candidate.resolve()
    return None


def resolve_natural_splits(manifest_path: Path) -> tuple[list[Path], list[Path]]:
    payload = json.loads(manifest_path.read_text(encoding="utf-8"))
    train, validation = [], []
    for item in payload.get("shards", []):
        if not isinstance(item, dict):
            continue
        value = item.get("file") or item.get("path")
        if not isinstance(value, str):
            continue
        path = resolve_path(manifest_path.parent, value)
        if path is None:
            continue
        split = str(item.get("split", "train")).lower()
        (validation if split in {"validation", "valid", "val", "dev"} else train).append(path)
    if not train or not validation:
        candidates = sorted(manifest_path.parent.glob("*.bin")) + sorted(manifest_path.parent.glob("*.npy"))
        validation = [p for p in candidates if any(x in p.name.lower() for x in ("validation", "valid", "val"))]
        train = [p for p in candidates if p not in validation]
    if not train or not validation:
        raise RuntimeError("Natural train/validation shard ayrimi yapilamadi")
    drive_paths = [p for p in train + validation if str(p).startswith("/content/drive/")]
    if drive_paths:
        raise RuntimeError(
            "DataLoader Drive'dan training will-not-run. Shard'lari /content/anka_shards "
            f"before continuing. First slow path: {drive_paths[0]}"
        )
    return train, validation


def available_ram_bytes() -> int:
    for line in Path("/proc/meminfo").read_text().splitlines():
        if line.startswith("MemAvailable:"):
            return int(line.split()[1]) * 1024
    raise RuntimeError("Available system RAM could not be read")


def load_tokens_to_ram(path: Path) -> np.ndarray:
    if path.suffix == ".npy":
        array = np.load(path, mmap_mode=None)
        array = np.asarray(array, dtype="<u2").reshape(-1)
    else:
        array = np.fromfile(path, dtype="<u2")
    if not array.flags.c_contiguous:
        array = np.ascontiguousarray(array)
    return array


class FastNaturalStore:
    def __init__(self, paths: list[Path], sequence_length: int):
        self.sequence_length = sequence_length
        self.arrays = []
        for index, path in enumerate(paths, 1):
            print(f"RAM train {index:02d}/{len(paths):02d}: {path.name}", flush=True)
            self.arrays.append(load_tokens_to_ram(path))
        self.windows = [
            np.lib.stride_tricks.sliding_window_view(array, sequence_length)
            for array in self.arrays
        ]
        self.usable = np.asarray([len(view) for view in self.windows], dtype=np.float64)
        if self.usable.sum() <= 0:
            raise RuntimeError("SEQ_LEN for yeterli natural token unavailable")
        self.probabilities = self.usable / self.usable.sum()

    def sample(self, rng: np.random.Generator, count: int) -> np.ndarray:
        shard_id = int(rng.choice(len(self.windows), p=self.probabilities))
        view = self.windows[shard_id]
        starts = rng.integers(0, len(view), size=count)
        return np.asarray(view[starts], dtype=np.int64)


class FastWorldStore:
    def __init__(self, manifest_path: Path, corpus: str = "paired"):
        payload = json.loads(manifest_path.read_text(encoding="utf-8"))
        assert payload["format"] == "worldpairs-fixed-uint16-v1"
        assert int(payload["sequence_length"]) == SEQ_LEN
        entry = payload["corpora"][corpus]
        self.sequence_length = int(payload["sequence_length"])
        self.rows = int(entry["rows"])
        flat_tokens = np.fromfile(
            manifest_path.parent / entry["tokens_file"], dtype="<u2"
        )
        self.tokens = flat_tokens.reshape(self.rows, self.sequence_length)
        self.lengths = np.fromfile(
            manifest_path.parent / entry["lengths_file"], dtype="<u2"
        )
        assert len(self.lengths) == self.rows
        self.mean_predicted_tokens = float(
            np.maximum(np.asarray(self.lengths, dtype=np.float64) - 1, 0).mean()
        )

    def sample(self, rng: np.random.Generator, count: int) -> tuple[np.ndarray, np.ndarray]:
        indices = rng.integers(0, self.rows, size=count)
        tokens = np.asarray(self.tokens[indices], dtype=np.int64)
        lengths = np.asarray(self.lengths[indices], dtype=np.int64)
        return tokens, lengths


TRAIN_SHARDS, VALIDATION_SHARDS = resolve_natural_splits(NATURAL_MANIFEST)

required_ram = sum(path.stat().st_size for path in TRAIN_SHARDS + VALIDATION_SHARDS)
world_payload = json.loads(WORLD_MANIFEST.read_text(encoding="utf-8"))
world_entry = world_payload["corpora"]["paired"]
required_ram += (WORLD_MANIFEST.parent / world_entry["tokens_file"]).stat().st_size
required_ram += (WORLD_MANIFEST.parent / world_entry["lengths_file"]).stat().st_size
free_ram = available_ram_bytes()
print("data RAM ihtiyaci:", f"{required_ram / 2**30:.2f} GiB")
print("Available RAM:", f"{free_ram / 2**30:.2f} GiB")
if required_ram > free_ram * 0.75:
    raise MemoryError(
        "Data exceeds 75% of available system RAM. "
        "Enable a high-RAM runtime or reduce the shard count."
    )

NATURAL_STORE = FastNaturalStore(TRAIN_SHARDS, SEQ_LEN)
WORLD_STORE = FastWorldStore(WORLD_MANIFEST, "paired")
VALIDATION_ARRAYS = []
for index, path in enumerate(VALIDATION_SHARDS, 1):
    print(f"RAM validation {index:02d}/{len(VALIDATION_SHARDS):02d}: {path.name}", flush=True)
    VALIDATION_ARRAYS.append(load_tokens_to_ram(path))


class FastBatchBuilder:
    def __init__(self, batch_size: int, world_ratio: float, seed: int):
        self.batch_size = batch_size
        self.world_ratio = world_ratio
        self.seed = seed
        self.world_row_probability = (
            0.0 if world_ratio == 0.0
            else world_ratio * (SEQ_LEN - 1) / WORLD_STORE.mean_predicted_tokens
        )
        if self.world_row_probability > 1.0:
            raise ValueError("Istenen WorldPairs token ratio row doluluguyla cannot be achieved")

    def _rng(self, step: int, stream: int) -> np.random.Generator:
        return np.random.default_rng(np.random.SeedSequence([self.seed, step, stream]))

    def build(self, step: int) -> tuple[torch.Tensor, dict]:
        natural_rng = self._rng(step, 1)
        world_rng = self._rng(step, 2)
        position_rng = self._rng(step, 3)
        rows = NATURAL_STORE.sample(natural_rng, self.batch_size)

        expected_before = math.floor(step * self.batch_size * self.world_row_probability)
        expected_after = math.floor((step + 1) * self.batch_size * self.world_row_probability)
        world_rows = expected_after - expected_before
        world_predicted = 0
        if world_rows:
            world_tokens, lengths = WORLD_STORE.sample(world_rng, world_rows)
            destinations = position_rng.choice(self.batch_size, size=world_rows, replace=False)
            selected = rows[destinations].copy()
            mask = np.arange(SEQ_LEN)[None, :] < lengths[:, None]
            np.copyto(selected, world_tokens, where=mask)
            rows[destinations] = selected
            world_predicted = int(np.maximum(lengths - 1, 0).sum())

        tensor = torch.from_numpy(np.ascontiguousarray(rows)).pin_memory()
        metrics = {
            "predicted_tokens": self.batch_size * (SEQ_LEN - 1),
            "world_predicted_tokens": world_predicted,
            "world_rows": world_rows,
        }
        return tensor, metrics


class OrderedBatchPrefetcher:
    def __init__(self, builder: FastBatchBuilder, start_step: int, depth: int = 6, workers: int = 2):
        self.builder = builder
        self.executor = ThreadPoolExecutor(max_workers=workers, thread_name_prefix="anka_data")
        self.pending = deque()
        self.next_submit = start_step
        for _ in range(depth):
            self._submit()

    def _submit(self) -> None:
        step = self.next_submit
        self.next_submit += 1
        self.pending.append((step, self.executor.submit(self.builder.build, step)))

    def next(self, expected_step: int) -> tuple[torch.Tensor, dict]:
        step, future = self.pending.popleft()
        if step != expected_step:
            raise RuntimeError(f"Prefetch sirasi was-corrupted: {step} != {expected_step}")
        result = future.result()
        self._submit()
        return result

    def close(self) -> None:
        for _, future in self.pending:
            future.cancel()
        self.executor.shutdown(wait=True, cancel_futures=True)
        self.pending.clear()


def fixed_validation(batch_size: int = 8, batches: int = 24) -> list[torch.Tensor]:
    arrays = [array for array in VALIDATION_ARRAYS if len(array) >= SEQ_LEN]
    result = []
    for batch_id in range(batches):
        array = arrays[batch_id % len(arrays)]
        view = np.lib.stride_tricks.sliding_window_view(array, SEQ_LEN)
        starts = np.linspace(
            0, len(view) - 1, batch_size + 2, dtype=np.int64
        )[1:-1]
        rows = np.asarray(view[starts], dtype=np.int64)
        result.append(torch.from_numpy(np.ascontiguousarray(rows)).pin_memory())
    return result


NATURAL_VALIDATION = fixed_validation()
print("Train shard      :", len(TRAIN_SHARDS))
print("Validation shard :", len(VALIDATION_SHARDS))
print("World rows       :", f"{WORLD_STORE.rows:,}")
print("World mean token :", f"{WORLD_STORE.mean_predicted_tokens:.1f}")


def benchmark_loader() -> None:
    builder = FastBatchBuilder(MICRO_BATCH, WORLD_RATIO, SEED + 99)
    prefetcher = OrderedBatchPrefetcher(builder, 0)
    for step in range(4):
        prefetcher.next(step)
    started = time.perf_counter()
    batches = 30
    for step in range(4, 4 + batches):
        batch, _ = prefetcher.next(step)
        assert batch.is_pinned() and batch.shape == (MICRO_BATCH, SEQ_LEN)
    elapsed = time.perf_counter() - started
    prefetcher.close()
    print(f"Loader: {1000 * elapsed / batches:.2f} ms/batch | {(batches * MICRO_BATCH * SEQ_LEN) / elapsed:,.0f} position/s")


benchmark_loader()


## 8. Define training, resume, validation, and cleanup logic

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# CELL 8 — Training engine, resume, validation, and cleanup
PEAK_LR = 2.5e-3
MIN_LR_RATIO = 0.10
WARMUP_RATIO = 0.02
WEIGHT_DECAY = 0.10
LOG_EVERY = 10
VALIDATE_EVERY = 500
SAVE_EVERY = 500


@dataclass(frozen=True)
class RunSpec:
    arm: str
    token_budget: int
    world_ratio: float
    seed: int = SEED

    @property
    def run_dir(self) -> Path:
        return OUTPUT_ROOT / self.arm / f"seed_{self.seed}"


def lr_at_tokens(seen_tokens: int, budget: int) -> float:
    warmup = max(1, int(budget * WARMUP_RATIO))
    if seen_tokens < warmup:
        return PEAK_LR * (seen_tokens + 1) / warmup
    progress = (seen_tokens - warmup) / max(1, budget - warmup)
    cosine = 0.5 * (1.0 + math.cos(math.pi * min(max(progress, 0.0), 1.0)))
    return PEAK_LR * (MIN_LR_RATIO + (1.0 - MIN_LR_RATIO) * cosine)


def pure_lm_metric(model: nn.Module, fallback: torch.Tensor) -> torch.Tensor:
    metrics = model.last_metrics
    value = metrics.get("final_loss")
    if value is None:
        value = metrics.get("lm_loss")
    return fallback if value is None else value


@torch.inference_mode()
def evaluate_natural(model: nn.Module) -> dict:
    model.eval()
    total_nll = 0.0
    total_tokens = 0
    started = time.perf_counter()
    for cpu_batch in NATURAL_VALIDATION:
        batch = cpu_batch.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            output = model(batch, labels=batch)
        metric = pure_lm_metric(model, output.loss)
        count = batch.shape[0] * (batch.shape[1] - 1)
        total_nll += float(metric) * count
        total_tokens += count
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    model.train()
    loss = total_nll / total_tokens
    return {
        "loss": loss,
        "perplexity": math.exp(min(loss, 20.0)),
        "tokens": total_tokens,
        "tokens_per_second": total_tokens / elapsed,
    }


def save_checkpoint(
    path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    spec: RunSpec,
    step: int,
    micro_step: int,
    seen_tokens: int,
    seen_world_tokens: int,
    history: list,
    final: bool,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp")
    payload = {
        "format": "anka-pairwise-v1",
        "spec": asdict(spec),
        "model_config": asdict(model.config),
        "model": model.state_dict(),
        "step": step,
        "micro_step": micro_step,
        "micro_batch": MICRO_BATCH,
        "grad_accum": GRAD_ACCUM,
        "seen_tokens": seen_tokens,
        "seen_world_tokens": seen_world_tokens,
        "history": history,
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": torch.cuda.get_rng_state_all(),
        "python_rng_state": random.getstate(),
    }
    if not final:
        payload["optimizer"] = optimizer.state_dict()
    torch.save(payload, temporary)
    temporary.replace(path)


def train_arm(spec: RunSpec) -> Path:
    if spec.arm not in BUILDERS:
        raise KeyError(f"Bilinmeyen kol: {spec.arm}")
    run_dir = spec.run_dir
    run_dir.mkdir(parents=True, exist_ok=True)
    latest_path = run_dir / "latest.pt"
    final_path = run_dir / "final.pt"
    result_path = run_dir / "training_result.json"
    if final_path.exists() and result_path.exists():
        print(f"{spec.arm}: completed run found, skipped.")
        return final_path

    clear_cuda()
    seed_everything(spec.seed)
    model = BUILDERS[spec.arm]().to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=PEAK_LR, betas=(0.9, 0.95), eps=1e-8,
        weight_decay=WEIGHT_DECAY, fused=True,
    )
    step = micro_step = seen_tokens = seen_world_tokens = 0
    history = []
    if latest_path.exists():
        checkpoint = torch.load(latest_path, map_location="cpu", weights_only=False)
        if checkpoint.get("format") != "anka-pairwise-v1":
            raise RuntimeError(f"Uyumsuz checkpoint: {latest_path}")
        if checkpoint["spec"] != asdict(spec):
            raise RuntimeError("Checkpoint RunSpec differs from this cell's RunSpec")
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        step = int(checkpoint["step"])
        micro_step = int(checkpoint.get("micro_step", step * GRAD_ACCUM))
        if int(checkpoint.get("micro_batch", MICRO_BATCH)) != MICRO_BATCH:
            raise RuntimeError("Resume micro_batch does not match the existing A100 preflight result")
        if int(checkpoint.get("grad_accum", GRAD_ACCUM)) != GRAD_ACCUM:
            raise RuntimeError("Resume grad_accum does not match the existing A100 preflight result")
        seen_tokens = int(checkpoint["seen_tokens"])
        seen_world_tokens = int(checkpoint["seen_world_tokens"])
        history = list(checkpoint.get("history", []))
        torch.set_rng_state(checkpoint["torch_rng_state"])
        torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state"])
        random.setstate(checkpoint["python_rng_state"])
        print(f"{spec.arm}: step={step:,}, token={seen_tokens:,} resuming from.")

    builder = FastBatchBuilder(MICRO_BATCH, spec.world_ratio, spec.seed + 17)
    prefetcher = OrderedBatchPrefetcher(builder, micro_step)
    model.train()
    optimizer.zero_grad(set_to_none=True)
    clear_cuda()
    torch.cuda.reset_peak_memory_stats()
    interval_start = time.perf_counter()
    interval_tokens = interval_world = interval_steps = 0
    interval_data = 0.0
    interval_losses = []
    started = time.time()

    print("\n" + "=" * 88)
    print(spec.arm, "training")
    print("=" * 88)
    print("parameter       :", f"{model.num_trainable_parameters():,}")
    print("Token budget   :", f"{spec.token_budget:,}")
    print("World ratio     :", f"{spec.world_ratio:.1%}")
    print("Micro/accum/seq :", MICRO_BATCH, "/", GRAD_ACCUM, "/", SEQ_LEN)

    batch = cpu_batch = output = raw_loss = loss = None
    try:
        while seen_tokens < spec.token_budget:
            lr = lr_at_tokens(seen_tokens, spec.token_budget)
            for group in optimizer.param_groups:
                group["lr"] = lr

            optimizer.zero_grad(set_to_none=True)
            step_losses = []
            for _ in range(GRAD_ACCUM):
                data_started = time.perf_counter()
                cpu_batch, batch_metrics = prefetcher.next(micro_step)
                batch = cpu_batch.to(DEVICE, non_blocking=True)
                interval_data += time.perf_counter() - data_started

                with torch.autocast("cuda", dtype=torch.bfloat16):
                    output = model(batch, labels=batch)
                    raw_loss = output.loss
                    loss = raw_loss / GRAD_ACCUM
                if not torch.isfinite(raw_loss):
                    raise FloatingPointError(
                        f"{spec.arm} micro_step={micro_step}: loss={raw_loss}"
                    )
                loss.backward()

                effective = int(batch_metrics["predicted_tokens"])
                world_tokens = int(batch_metrics["world_predicted_tokens"])
                seen_tokens += effective
                seen_world_tokens += world_tokens
                micro_step += 1
                interval_tokens += effective
                interval_world += world_tokens
                step_losses.append(raw_loss.detach())
                del batch, cpu_batch, output, raw_loss, loss

            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(), 1.0, error_if_nonfinite=True
            )
            optimizer.step()
            step += 1
            interval_steps += 1
            interval_losses.append(torch.stack(step_losses).mean())

            if step % LOG_EVERY == 0:
                torch.cuda.synchronize()
                elapsed = time.perf_counter() - interval_start
                record = {
                    "step": step,
                    "seen_tokens": seen_tokens,
                    "loss": float(torch.stack(interval_losses).mean()),
                    "lr": lr,
                    "grad_norm": float(grad_norm),
                    "tokens_per_second": interval_tokens / elapsed,
                    "world_ratio": interval_world / max(1, interval_tokens),
                    "data_wait_ms_per_step": 1000 * interval_data / max(1, interval_steps),
                    "peak_vram_gib": torch.cuda.max_memory_allocated() / 2**30,
                }
                history.append(record)
                print(
                    f"[{spec.arm}] step={step:05d} token={seen_tokens:,}/{spec.token_budget:,} "
                    f"loss={record['loss']:.4f} lr={lr:.3e} grad={record['grad_norm']:.3f} "
                    f"tok/s={record['tokens_per_second']:,.0f} world={record['world_ratio']:.2%} "
                    f"data={record['data_wait_ms_per_step']:.2f}ms peak={record['peak_vram_gib']:.2f}GiB"
                )
                interval_start = time.perf_counter()
                interval_tokens = interval_world = interval_steps = 0
                interval_data = 0.0
                interval_losses = []

            validate_now = step % VALIDATE_EVERY == 0
            save_now = step % SAVE_EVERY == 0
            if validate_now:
                validation = evaluate_natural(model)
                print(f"[{spec.arm}] natural_val={validation['loss']:.6f} ppl={validation['perplexity']:.2f}")
            if save_now:
                save_checkpoint(
                    latest_path, model, optimizer, spec, step, micro_step,
                    seen_tokens, seen_world_tokens, history, final=False,
                )
            if validate_now or save_now:
                torch.cuda.synchronize()
                interval_start = time.perf_counter()
    except BaseException:
        prefetcher.close()
        optimizer.zero_grad(set_to_none=True)
        batch = cpu_batch = output = raw_loss = loss = None
        del model, optimizer, builder, prefetcher
        clear_cuda()
        raise
    else:
        prefetcher.close()

    validation = evaluate_natural(model)
    save_checkpoint(
        final_path, model, optimizer, spec, step, micro_step,
        seen_tokens, seen_world_tokens, history, final=True,
    )
    result = {
        "arm": spec.arm,
        "parameters": model.num_trainable_parameters(),
        "token_budget": spec.token_budget,
        "seen_tokens": seen_tokens,
        "seen_world_tokens": seen_world_tokens,
        "actual_world_ratio": seen_world_tokens / seen_tokens,
        "natural_validation": validation,
        "wall_seconds": time.time() - started,
        "peak_vram_gib": torch.cuda.max_memory_allocated() / 2**30,
        "history": history,
    }
    result_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(json.dumps({k: v for k, v in result.items() if k != "history"}, indent=2))
    del model, optimizer, builder
    clear_cuda()
    return final_path


## 9. Train the three pairwise experiment arms

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# CELL 9 — Run the three pairwise arms sequentially
SPECS = [
    RunSpec("RM3", TOKEN_BUDGET, 0.0),
    RunSpec("RW3", TOKEN_BUDGET, WORLD_RATIO),
    RunSpec("MW3", TOKEN_BUDGET, WORLD_RATIO),
]

FINAL_PATHS = {}
for spec in SPECS:
    FINAL_PATHS[spec.arm] = train_arm(spec)


## 10. Evaluate held-out WorldPairs behavior

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# CELL 10 — Held-out WorldPairs evaluation
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
PAD_ID = tokenizer.token_to_id("<pad>")
assert PAD_ID is not None


def load_suite(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line]


def encode_candidate(prompt: str, candidate: str) -> dict:
    encoding = tokenizer.encode(prompt + " " + candidate + ".", add_special_tokens=False)
    boundary = len(prompt)
    positions = [i for i, (_start, end) in enumerate(encoding.offsets) if end > boundary]
    if not positions or min(positions) == 0:
        raise ValueError("Could not derive candidate token boundary")
    return {"ids": encoding.ids, "target_positions": positions}


@torch.inference_mode()
def score_tasks(model: nn.Module, tasks: list[dict], batch_size: int = 12) -> list[dict]:
    prepared = []
    for task_id, task in enumerate(tasks):
        prepared.append({"task_id": task_id, **task, **encode_candidate(task["prompt"], task["candidate"])})
    prepared.sort(key=lambda item: len(item["ids"]))
    results = [None] * len(tasks)
    model.eval()
    for start in range(0, len(prepared), batch_size):
        chunk = prepared[start:start + batch_size]
        maximum = max(len(item["ids"]) for item in chunk)
        batch = torch.full((len(chunk), maximum), PAD_ID, dtype=torch.long, device=DEVICE)
        for row, item in enumerate(chunk):
            batch[row, :len(item["ids"])] = torch.tensor(item["ids"], device=DEVICE)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = model(batch).logits
        for row, item in enumerate(chunk):
            positions = item["target_positions"]
            predictor = torch.tensor([p - 1 for p in positions], device=DEVICE)
            targets = torch.tensor([item["ids"][p] for p in positions], device=DEVICE)
            selected = logits[row].index_select(0, predictor).float()
            token_scores = F.log_softmax(selected, dim=-1).gather(1, targets[:, None])[:, 0]
            results[item["task_id"]] = {
                **{k: item[k] for k in ("record_index", "member", "candidate_index")},
                "raw_score": float(token_scores.sum()),
                "normalized_score": float(token_scores.mean()),
            }
        del batch, logits
    return results


def worldpair_scores(model: nn.Module, records: list[dict]) -> dict:
    tasks = []
    for record_index, record in enumerate(records):
        for member in ("a", "b"):
            for candidate_index, candidate in enumerate(record["candidate_texts"]):
                tasks.append({
                    "record_index": record_index,
                    "member": member,
                    "candidate_index": candidate_index,
                    "candidate": candidate,
                    "prompt": record[f"prompt_{member}"],
                })
    scored = score_tasks(model, tasks)
    grouped = defaultdict(list)
    for item in scored:
        grouped[(item["record_index"], item["member"])].append(item)

    rows = []
    for index, record in enumerate(records):
        row = {"split": record["split"], "family_id": record["family_id"]}
        for member in ("a", "b"):
            options = sorted(grouped[(index, member)], key=lambda item: item["candidate_index"])
            correct_id = record[f"answer_id_{member}"]
            correct_index = record["candidate_answer_ids"].index(correct_id)
            for metric in ("raw", "normalized"):
                values = [item[f"{metric}_score"] for item in options]
                row[f"{member}_{metric}"] = int(np.argmax(values)) == correct_index
        for metric in ("raw", "normalized"):
            row[f"pair_{metric}"] = row[f"a_{metric}"] and row[f"b_{metric}"]
        rows.append(row)

    result = {}
    for metric in ("raw", "normalized"):
        per_split = []
        for split in sorted({row["split"] for row in rows}):
            subset = [row for row in rows if row["split"] == split]
            per_split.append(float(np.mean([row[f"pair_{metric}"] for row in subset])))
        result[f"macro_pair_accuracy_{metric}"] = float(np.mean(per_split))
    return result


## 11. Compare checkpoints and save the final report

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# CELL 11 — Final comparison, samples, and report
@torch.inference_mode()
def generate(model: nn.Module, prompt: str, max_new_tokens: int = 100, temperature: float = 0.8, top_k: int = 40) -> str:
    ids = tokenizer.encode(prompt).ids
    x = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    model.eval()
    for _ in range(max_new_tokens):
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = model(x[:, -model.config.max_seq_len:]).logits[:, -1].float()
        logits /= temperature
        values, indices = torch.topk(logits, min(top_k, logits.shape[-1]))
        filtered = torch.full_like(logits, -float("inf"))
        filtered.scatter_(1, indices, values)
        next_id = torch.multinomial(torch.softmax(filtered, dim=-1), 1)
        x = torch.cat([x, next_id], dim=1)
    return tokenizer.decode(x[0].tolist(), skip_special_tokens=True)


records = load_suite(PROOF_SUITE)
prompts = [
    "The future of",
    "Science is important because",
    "If a box is placed on a table",
]
reports = []
for spec in SPECS:
    clear_cuda()
    checkpoint = torch.load(FINAL_PATHS[spec.arm], map_location="cpu", weights_only=False)
    model = BUILDERS[spec.arm]().to(DEVICE)
    model.load_state_dict(checkpoint["model"])
    natural = evaluate_natural(model)
    world = worldpair_scores(model, records)
    samples = [{"prompt": p, "completion": generate(model, p)} for p in prompts]
    training = json.loads((spec.run_dir / "training_result.json").read_text(encoding="utf-8"))
    recent_speed = float(np.mean([
        row["tokens_per_second"] for row in training["history"][-50:]
    ]))
    report = {
        "arm": spec.arm,
        "parameters": model.num_trainable_parameters(),
        "seen_tokens": checkpoint["seen_tokens"],
        "actual_world_ratio": training["actual_world_ratio"],
        "natural_loss": natural["loss"],
        "perplexity": natural["perplexity"],
        **world,
        "recent_training_tokens_per_second": recent_speed,
        "samples": samples,
    }
    if "reflect_gate" in model.last_metrics:
        report["reflect_gate"] = float(model.last_metrics["reflect_gate"])
    if "router_entropy" in model.last_metrics:
        report["router_entropy"] = float(model.last_metrics["router_entropy"])
        report["stage_loads"] = model.last_metrics["stage_loads"].cpu().tolist()
    reports.append(report)
    print(json.dumps(report, indent=2, ensure_ascii=False))
    del model, checkpoint
    clear_cuda()

table = pd.DataFrame([{k: v for k, v in row.items() if k not in {"samples", "stage_loads"}} for row in reports])
table = table.sort_values(
    ["macro_pair_accuracy_normalized", "natural_loss"],
    ascending=[False, True],
).reset_index(drop=True)
display(table)

REPORT_JSON = OUTPUT_ROOT / "pairwise_comparison.json"
REPORT_CSV = OUTPUT_ROOT / "pairwise_comparison.csv"
REPORT_JSON.write_text(json.dumps(reports, indent=2, ensure_ascii=False), encoding="utf-8")
table.to_csv(REPORT_CSV, index=False)
print("report:", REPORT_JSON)
